In [1]:
import sys

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr3"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
def generate_random_sequence(length, alphabet=np.array(list("ACGT"))):
    percs = np.random.dirichlet(alpha=np.full(fill_value=5.0, shape=(4,)))
    seq = np.random.choice(alphabet, size=(length,), replace=True, p=percs)
    return "".join(seq)

In [6]:
if utr_type == "utr5":
    utr_length = 50
    cell_types = ["c1", "c2", "c4", "c6", "c17"]
elif utr_type == "utr3":
    utr_length = 240
    cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]

In [7]:
from itertools import product

n_seqs = 1_000_000
np.random.seed(777)
df = pd.DataFrame(product([generate_random_sequence(utr_length) for _ in range(n_seqs)], cell_types), columns=["seq", "cell_type"])
df.head()

,seq,cell_type
0,AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGA...,c1
1,AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGA...,c2
2,AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGA...,c4
3,AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGA...,c6
4,AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGA...,c13


In [8]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

6

In [9]:
batch_size = 1024

In [10]:
num_workers = 32

In [11]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [12]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [13]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding
ckpt["hyper_parameters"]["model_class"]

loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

<All keys matched successfully>

In [14]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [15]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [16]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")

In [17]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)

In [18]:
result = pd.DataFrame(
    pred_mass_center,
    columns=pd.MultiIndex.from_product([["pred_mass_center"], cell_types]),
    index=df.iloc[::num_classes]["seq"]
)
result["pred_mass_center"].head()

,c1,c2,c4,c6,c13,c17
seq,,,,,,
AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGAGTACGCGAATTGTGGTATCTCCTTGCAGGATTACCTTTATTCTGTTCATTGTATGGCGTGAGGATGGTATTTATACTAAGTAGGTCTTGGTTTGTCGTGGAAAGTATAGGAAGAATAATGAGGTCGGTGGTAACCATATTCGTATGAGTGAGCTACGAGGTGTTCGGGACGAACTGCTTGGATGTGAGAGTGTA,2.427997,2.255967,2.403729,2.344373,2.364047,2.219207
AAAAAGAAAAAGAAAAATAAATAAGAATATGAAACGCGTAGATAGAATTATAACCAACGTCTATAAGCAAAGGTAATAGATTTTTTTACAACACAGAAGTCGTTACTCCAACAAAGTGACTGTGTAAAACTACTTATACAGCATAATCGCACAAATAAAACACGGGACTGAGAACAGCAGAGTCTAATATATCGTTATAAACACATAGCAACATCTTATTAAACATTACTAGAACGAATC,2.399186,2.643656,2.532826,2.682189,2.584194,2.710167
TTCTATCCCCAATTCGATATTCCCTCCTATGATTCGATACTGTATCAGTGTCTGTTGAAAGAACTGTCAGTTAAGCCTATGATAGTCTCCATAAATGAATGGTAGCTACAATTGATCCATATTGTTGGCATAATCAAGGAAACCTGTATCTCGTCTCTTTGCGAATGCCATGGGTAATTAGACCATAATAATACGTGACTTCCAGAGTGCCTTTTATGCTCATGACGCGTACCTATGTAT,2.610810,2.581765,2.558408,2.627713,2.525304,2.600102
TCTAATAATTTGTGAAGTTCAAAACTTAACACAAAGAAAAAACCTAGTTAAATGTAGTACAATATCTCATGAATAGTGTAGATAAGTGGAGGATGGAACTCATCGGCGCAACTGAATGAAAAAGAGATTGACATACATGCAATGGTAAAGTACGCTGAAAAATCAATATTCAAATAAATTATTGCTAAGTTATTTAGAAAGAAAGAGGTACAAGCCTAACATATATAGAATTGAAAAACA,2.452590,2.537662,2.563846,2.492126,2.495237,2.505347
ACTGCCGGCGCGCGCCTAAAAGAGGCCGGGGGCGTTGGGGATGCCGCGTCGCCCAGCAGGGCACGGCACAACTGGGAGGCGGGGGCTGGAGCTCTTCGGAGGTCAGGTGGGCCCTAACATGGCGCCGGCTAGGGGGGGGACAGACACGCGCCAGCTACGGGGGAGCCGGCCCCGCGAGGTGGCTCACGCAGCGAGGCACATGGGGAGAGCGTGTGCGCGAGCCGGGTGCGCATGGAGGCA,1.943647,2.074082,2.197204,2.133068,2.306998,2.015381


In [19]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["pred_mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [20]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c17_022,c17_023,c17_024,c17_025,c17_026,c17_027,c17_028,c17_029,c17_030,c17_031
seq,,,,,,,,,,,,,,,,,,,,,
AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGAGTACGCGAATTGTGGTATCTCCTTGCAGGATTACCTTTATTCTGTTCATTGTATGGCGTGAGGATGGTATTTATACTAAGTAGGTCTTGGTTTGTCGTGGAAAGTATAGGAAGAATAATGAGGTCGGTGGTAACCATATTCGTATGAGTGAGCTACGAGGTGTTCGGGACGAACTGCTTGGATGTGAGAGTGTA,0.002775,0.101027,0.166487,0.125139,0.152757,0.053387,0.120255,0.190259,0.037895,0.030835,...,0.263547,0.312517,0.175730,0.099520,0.001215,-0.005591,0.026959,0.001136,0.111191,-0.006003
AAAAAGAAAAAGAAAAATAAATAAGAATATGAAACGCGTAGATAGAATTATAACCAACGTCTATAAGCAAAGGTAATAGATTTTTTTACAACACAGAAGTCGTTACTCCAACAAAGTGACTGTGTAAAACTACTTATACAGCATAATCGCACAAATAAAACACGGGACTGAGAACAGCAGAGTCTAATATATCGTTATAAACACATAGCAACATCTTATTAAACATTACTAGAACGAATC,0.066997,0.120147,0.002547,0.133043,0.070987,0.312444,-0.061147,-0.006088,0.204470,0.210711,...,0.081906,0.048840,0.076260,0.145465,0.175509,0.152068,0.074445,0.167128,0.130040,0.126432
TTCTATCCCCAATTCGATATTCCCTCCTATGATTCGATACTGTATCAGTGTCTGTTGAAAGAACTGTCAGTTAAGCCTATGATAGTCTCCATAAATGAATGGTAGCTACAATTGATCCATATTGTTGGCATAATCAAGGAAACCTGTATCTCGTCTCTTTGCGAATGCCATGGGTAATTAGACCATAATAATACGTGACTTCCAGAGTGCCTTTTATGCTCATGACGCGTACCTATGTAT,0.109702,0.149064,0.046762,0.097995,0.063150,0.051001,0.157237,0.183697,0.119873,0.075379,...,0.120137,0.040993,0.034569,0.124216,0.151456,0.147173,0.136023,0.130215,0.164837,0.133407
TCTAATAATTTGTGAAGTTCAAAACTTAACACAAAGAAAAAACCTAGTTAAATGTAGTACAATATCTCATGAATAGTGTAGATAAGTGGAGGATGGAACTCATCGGCGCAACTGAATGAAAAAGAGATTGACATACATGCAATGGTAAAGTACGCTGAAAAATCAATATTCAAATAAATTATTGCTAAGTTATTTAGAAAGAAAGAGGTACAAGCCTAACATATATAGAATTGAAAAACA,0.149876,0.093226,0.026301,0.100091,0.085777,0.215382,0.123668,0.018274,0.206299,0.168044,...,0.093899,0.054216,0.036046,0.079095,0.120127,0.106859,0.079039,0.129122,0.117390,0.105493
ACTGCCGGCGCGCGCCTAAAAGAGGCCGGGGGCGTTGGGGATGCCGCGTCGCCCAGCAGGGCACGGCACAACTGGGAGGCGGGGGCTGGAGCTCTTCGGAGGTCAGGTGGGCCCTAACATGGCGCCGGCTAGGGGGGGGACAGACACGCGCCAGCTACGGGGGAGCCGGCCCCGCGAGGTGGCTCACGCAGCGAGGCACATGGGGAGAGCGTGTGCGCGAGCCGGGTGCGCATGGAGGCA,-0.140775,0.007953,0.281581,0.194234,0.286187,0.158981,-0.136274,0.223772,0.076208,0.045015,...,0.194787,0.390578,0.370095,0.061111,-0.079984,-0.083830,-0.084586,-0.065421,-0.036351,-0.055613
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGCGAGGTAACGCCTTACCCCTTCTGGACCCCACCAATATGGCCTCTTCATGACAGAGTGCCTCTAGTCGCCCGTTGACCCAGATTCTCCAATATTCGTCCATTCGACTATCCTTGTACCCAGCTCGCCCCTTAAGCCCCGCCCGTTCCTCCGGTCGCTCGCTTTCACCCTTTTGTCCCCAAGAGAATACGGCCCATCCTCGGGATTCTTGTCAACGGTTACAGTAGACTGAACCCCG,0.034396,0.143378,0.018254,0.092340,0.059302,0.148254,0.042324,0.136998,0.160459,0.120798,...,0.135760,0.072726,0.076746,0.144403,0.164517,0.154064,0.107160,0.117459,0.162263,0.095063
CCCATATCTTAGCGTTCCTGGAACGGCTAGGGGCAGGGGGGTGGACAGTCCAAGCGTAGGCTGTTCTGGTTTGCGCCGGTTGGGTCAAGTGCTAATTTCGACGCTCCACTCACATCCTCTAGTCCGTCTTGTCTGAAACGATACTTTGGTGTCCGTTCCCACGTCCTTTACACAAACATGTTTGAGTATTTTGTCCCGGTCGCTTCCTCGCTCGGTAAGCAATAATACAGCCGCTCCTCC,0.071036,0.146253,0.104511,0.121146,0.089192,0.042150,0.102990,0.216881,0.082186,0.071011,...,0.138580,0.166784,0.130446,0.097461,0.106081,0.107983,0.107042,0.068600,0.133464,0.087738
ATCTTACTGTCCGGTGAACAAAGTGATAAATCGTGAATAAAGTTTTATTATAAATCTCTATCCTGCACCTGATGGTATCTTCTTGAGGCATGATACCCAATTTGGACGATAATGAGGAAATATTGAAGAATTAGACGTAGATTATCTTATAGTTCATCGGGGATATTTATAAGAAGACTTTTGAGTAGGCGGCTACTAAAGGCTGGAGAGGCGTCGTGAAACTTTAATAGTTGGCTGGAG,0.144262,0.102105,0.096432,0.126633,0.109819,0.071207,0.144352,0.136515,0.110846,0.103127,...,0.157693,0.134243,0.067537,0.091078,0.097326,0.081402,0.083153,0.088173,0.135332,0.099642


### Counting k-mers

In [21]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [22]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [23]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAGGTGTGCGCCGCGTGCATCGGGGAGTTACGTGTCGGGTATGGGAGTACGCGAATTGTGGTATCTCCTTGCAGGATTACCTTTATTCTGTTCATTGTATGGCGTGAGGATGGTATTTATACTAAGTAGGTCTTGGTTTGTCGTGGAAAGTATAGGAAGAATAATGAGGTCGGTGGTAACCATATTCGTATGAGTGAGCTACGAGGTGTTCGGGACGAACTGCTTGGATGTGAGAGTGTA,53,32,81,74,10,8,16,18,4,4,...,6,3,5,4,8,9,4,4,6,3
AAAAAGAAAAAGAAAAATAAATAAGAATATGAAACGCGTAGATAGAATTATAACCAACGTCTATAAGCAAAGGTAATAGATTTTTTTACAACACAGAAGTCGTTACTCCAACAAAGTGACTGTGTAAAACTACTTATACAGCATAATCGCACAAATAAAACACGGGACTGAGAACAGCAGAGTCTAATATATCGTTATAAACACATAGCAACATCTTATTAAACATTACTAGAACGAATC,112,39,34,55,48,23,18,23,19,2,...,3,3,3,0,0,2,8,0,0,5
TTCTATCCCCAATTCGATATTCCCTCCTATGATTCGATACTGTATCAGTGTCTGTTGAAAGAACTGTCAGTTAAGCCTATGATAGTCTCCATAAATGAATGGTAGCTACAATTGATCCATATTGTTGGCATAATCAAGGAAACCTGTATCTCGTCTCTTTGCGAATGCCATGGGTAATTAGACCATAATAATACGTGACTTCCAGAGTGCCTTTTATGCTCATGACGCGTACCTATGTAT,66,51,43,80,17,9,10,30,12,15,...,3,6,7,4,3,7,3,5,5,3
TCTAATAATTTGTGAAGTTCAAAACTTAACACAAAGAAAAAACCTAGTTAAATGTAGTACAATATCTCATGAATAGTGTAGATAAGTGGAGGATGGAACTCATCGGCGCAACTGAATGAAAAAGAGATTGACATACATGCAATGGTAAAGTACGCTGAAAAATCAATATTCAAATAAATTATTGCTAAGTTATTTAGAAAGAAAGAGGTACAAGCCTAACATATATAGAATTGAAAAACA,108,29,42,61,50,13,18,26,15,2,...,1,2,7,2,3,3,5,2,4,2
ACTGCCGGCGCGCGCCTAAAAGAGGCCGGGGGCGTTGGGGATGCCGCGTCGCCCAGCAGGGCACGGCACAACTGGGAGGCGGGGGCTGGAGCTCTTCGGAGGTCAGGTGGGCCCTAACATGGCGCCGGCTAGGGGGGGGACAGACACGCGCCAGCTACGGGGGAGCCGGCCCCGCGAGGTGGCTCACGCAGCGAGGCACATGGGGAGAGCGTGTGCGCGAGCCGGGTGCGCATGGAGGCA,40,68,108,24,5,11,19,4,16,15,...,2,1,0,4,8,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGCGAGGTAACGCCTTACCCCTTCTGGACCCCACCAATATGGCCTCTTCATGACAGAGTGCCTCTAGTCGCCCGTTGACCCAGATTCTCCAATATTCGTCCATTCGACTATCCTTGTACCCAGCTCGCCCCTTAAGCCCCGCCCGTTCCTCCGGTCGCTCGCTTTCACCCTTTTGTCCCCAAGAGAATACGGCCCATCCTCGGGATTCTTGTCAACGGTTACAGTAGACTGAACCCCG,44,88,45,63,8,14,11,11,13,40,...,7,5,3,2,2,3,3,8,5,4
CCCATATCTTAGCGTTCCTGGAACGGCTAGGGGCAGGGGGGTGGACAGTCCAAGCGTAGGCTGTTCTGGTTTGCGCCGGTTGGGTCAAGTGCTAATTTCGACGCTCCACTCACATCCTCTAGTCCGTCTTGTCTGAAACGATACTTTGGTGTCCGTTCCCACGTCCTTTACACAAACATGTTTGAGTATTTTGTCCCGGTCGCTTCCTCGCTCGGTAAGCAATAATACAGCCGCTCCTCC,43,68,60,69,11,12,11,9,14,18,...,4,5,2,2,5,5,2,5,6,7
ATCTTACTGTCCGGTGAACAAAGTGATAAATCGTGAATAAAGTTTTATTATAAATCTCTATCCTGCACCTGATGGTATCTTCTTGAGGCATGATACCCAATTTGGACGATAATGAGGAAATATTGAAGAATTAGACGTAGATTATCTTATAGTTCATCGGGGATATTTATAAGAAGACTTTTGAGTAGGCGGCTACTAAAGGCTGGAGAGGCGTCGTGAAACTTTAATAGTTGGCTGGAG,75,33,57,75,23,9,17,26,5,5,...,3,6,10,1,5,1,8,2,5,7


In [24]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_dirichlet.parquet")

---